 ### Finviz Stock & ETF Data Scraper

 Compatible with both local PC (VS Code) and Google Colab environments.

In [1]:
import datetime
from io import StringIO
from pathlib import Path
import random
import sys
import time
import zipfile

from curl_cffi import requests
import pandas as pd
import pytz

# Detect execution environment
IS_COLAB = "google.colab" in sys.modules

if IS_COLAB:
    output_dir = Path(".")
    print("Running in Google Colab environment.")
else:
    output_dir = Path.home() / "Downloads"
    print(f"Running on local PC. Output directory: {output_dir.resolve()}")

output_dir.mkdir(parents=True, exist_ok=True)

Running in Google Colab environment.


In [2]:
url_all_columns = "0,1,2,79,3,4,5,129,6,7,8,9,10,11,12,13,73,74,14,130,131,15,16,77,17,18,19,20,21,23,22,132,133,82,78,127,128,24,25,85,26,27,28,29,30,31,84,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,134,125,126,59,68,70,80,83,76,60,61,62,63,64,67,69,81,86,87,88,65,66,103,100,107,108,109,112,113,114,115,116,117,120,121,122,105"

url_mktcap = "https://finviz.com/screener.ashx?v=152&ft=4&o=-marketcap&c="
url_mktcap_rows = [
    "&r=1",
    "&r=21",
    "&r=41",
    "&r=61",
    "&r=81",
    "&r=101",
    "&r=121",
    "&r=141",
    "&r=161",
    "&r=181",
    "&r=201",
    "&r=221",
    "&r=241",
    "&r=261",
    "&r=281",
    "&r=301",
    "&r=321",
    "&r=341",
    "&r=361",
    "&r=381",
    "&r=401",
    "&r=421",
    "&r=441",
    "&r=461",
    "&r=481",
    "&r=501",
    "&r=521",
    "&r=541",
    "&r=561",
    "&r=581",
    "&r=601",
    "&r=621",
    "&r=641",
    "&r=661",
    "&r=681",
    "&r=701",
    "&r=721",
    "&r=741",
    "&r=761",
    "&r=781",
    "&r=801",
    "&r=821",
    "&r=841",
    "&r=861",
    "&r=881",
    "&r=901",
    "&r=921",
    "&r=941",
    "&r=961",
    "&r=981",
    "&r=1001",
    "&r=1021",
    "&r=1041",
    "&r=1061",
    "&r=1081",
    "&r=1101",
    "&r=1121",
    "&r=1141",
]

url_etfs = "https://finviz.com/screener.ashx?v=152&ft=4&o=-e.assetsundermanagement&c="
url_etfs_rows = [
    "&r=1",
    "&r=21",
    "&r=41",
    "&r=61",
    "&r=81",
    "&r=101",
    "&r=121",
    "&r=141",
    "&r=161",
    "&r=181",
    "&r=201",
    "&r=221",
    "&r=241",
    "&r=261",
    "&r=281",
    "&r=301",
    "&r=321",
    "&r=341",
    "&r=361",
    "&r=381",
]

urls_mktcap = [f"{url_mktcap}{url_all_columns}{row}" for row in url_mktcap_rows]
urls_etfs = [f"{url_etfs}{url_all_columns}{row}" for row in url_etfs_rows]

urls = urls_mktcap + urls_etfs
shuffled_urls = random.sample(urls, len(urls))
print(f"Generated {len(shuffled_urls)} total URLs to scrape.")

Generated 78 total URLs to scrape.


In [3]:
def scrape_finviz_curl(url: str) -> pd.DataFrame | None:
    try:
        response = requests.get(url, impersonate="chrome120")
        if response.status_code != 200:
            print(f"Failed HTTP Request. Status Code: {response.status_code}")
            return None

        dfs = pd.read_html(StringIO(response.text))
        screener_df = max(dfs, key=len)

        if "Ticker" not in screener_df.columns:
            screener_df.columns = screener_df.iloc[0]
            screener_df = screener_df[1:]

        return screener_df.reset_index(drop=True)

    except Exception as e:
        print(f"Error scraping URL ({url}): {e}")
        return None

In [4]:
total_urls = len(shuffled_urls)
df = pd.DataFrame()

print(f"Starting extraction across {total_urls} pages...")

# for idx, url in enumerate(shuffled_urls[0:3], start=1):
for idx, url in enumerate(shuffled_urls, start=1):
    delay = random.uniform(2, 4)
    print(f"[{idx}/{total_urls}] Downloading page... Sleeping {delay:.2f}s")
    time.sleep(delay)

    df_temp = scrape_finviz_curl(url)
    if df_temp is not None:
        df = pd.concat([df, df_temp], ignore_index=True)
    else:
        print(f"Skipped page due to download failure: {url}")

Starting extraction across 78 pages...
[1/78] Downloading page... Sleeping 2.41s
[2/78] Downloading page... Sleeping 3.34s
[3/78] Downloading page... Sleeping 3.77s
[4/78] Downloading page... Sleeping 2.10s
[5/78] Downloading page... Sleeping 2.52s
[6/78] Downloading page... Sleeping 2.79s
[7/78] Downloading page... Sleeping 2.68s
[8/78] Downloading page... Sleeping 2.73s
[9/78] Downloading page... Sleeping 3.61s
[10/78] Downloading page... Sleeping 2.05s
[11/78] Downloading page... Sleeping 2.02s
[12/78] Downloading page... Sleeping 3.38s
[13/78] Downloading page... Sleeping 2.24s
[14/78] Downloading page... Sleeping 2.99s
[15/78] Downloading page... Sleeping 3.38s
[16/78] Downloading page... Sleeping 2.64s
[17/78] Downloading page... Sleeping 3.03s
[18/78] Downloading page... Sleeping 2.11s
[19/78] Downloading page... Sleeping 3.80s
[20/78] Downloading page... Sleeping 3.27s
[21/78] Downloading page... Sleeping 3.30s
[22/78] Downloading page... Sleeping 3.87s
[23/78] Downloading page

In [5]:
if not df.empty and "Ticker" in df.columns:
    tickers_str = df["Ticker"].astype(str)

    # 1. Sanity Check: Ensure tickers have >= 2 characters before stripping
    short_tickers = tickers_str[tickers_str.str.len() < 2]
    if not short_tickers.empty:
        print(
            f"Warning: {len(short_tickers)} ticker(s) had fewer than 2 characters before modification:"
        )
        print(short_tickers.tolist())
    else:
        print("Sanity Check Passed: All tickers contain at least 2 characters.")

    # Remove the prefix character from tickers
    df["Ticker"] = tickers_str.str[1:]

    # 2. Check presence of SPY and AAPL after stripping
    unique_tickers = set(df["Ticker"])
    for check_symbol in ["SPY", "AAPL"]:
        found = check_symbol in unique_tickers
        print(f"Verification Check - Symbol '{check_symbol}' found: {found}")

    # Numeric conversion for appropriate data columns
    for col in df.columns:
        if col == "Ticker":
            continue
        converted = pd.to_numeric(df[col], errors="coerce")
        if converted.notna().any():
            df[col] = converted

Sanity Check Passed: All tickers contain at least 2 characters.
Verification Check - Symbol 'SPY' found: True
Verification Check - Symbol 'AAPL' found: True


In [6]:
pst = pytz.timezone("America/Los_Angeles")
current_date_pst = datetime.datetime.now(pst).strftime("%Y-%m-%d")

parquet_filename = f"df_finviz_{current_date_pst}_stocks_etfs.parquet"
csv_filename = f"ticker_{current_date_pst}_stocks_etfs.csv"
zip_filename = f"{current_date_pst}_combined_files.zip"

parquet_path = output_dir / parquet_filename
csv_path = output_dir / csv_filename
zip_path = output_dir / zip_filename

# Save parquet and ticker CSV
df.to_parquet(parquet_path, engine="pyarrow", compression="zstd")
if "Ticker" in df.columns:
    df[["Ticker"]].to_csv(csv_path, header=False, index=False)

# Archive into ZIP
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zipf:
    zipf.write(parquet_path, arcname=parquet_filename)
    if csv_path.exists():
        zipf.write(csv_path, arcname=csv_filename)

print(f"Files successfully generated and compressed at:\n{zip_path.resolve()}")

# Clean up loose files
parquet_path.unlink(missing_ok=True)
csv_path.unlink(missing_ok=True)

# Trigger automatic download to local PC Downloads folder if running in Colab
if IS_COLAB:
    from google.colab import files

    print("Initiating browser download to PC Downloads folder...")
    files.download(str(zip_path))



Files successfully generated and compressed at:
/content/2026-07-21_combined_files.zip
Initiating browser download to PC Downloads folder...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [9]:
df

,No.,Ticker,Company,Index,Sector,Industry,Country,Exchange,Market Cap,P/E,...,Flows 1M,Flows% 1M,Flows 3M,Flows% 3M,Flows YTD,Flows% YTD,Return% 1Y,Return% 3Y,Return% 5Y,Tags
0,901,WTFC,Wintrust Financial Corp,-,Financial,Banks - Regional,USA,NASD,10.72B,12.77,...,NaN,-,NaN,-,-,-,-,-,-,-
1,902,AM,Antero Midstream Corp,-,Energy,Oil & Gas Midstream,USA,NYSE,10.69B,26.38,...,NaN,-,NaN,-,-,-,-,-,-,-
2,903,CART,Maplebear Inc,-,Consumer Cyclical,Internet Retail,USA,NASD,10.68B,25.49,...,NaN,-,NaN,-,-,-,-,-,-,-
3,904,SCI,Service Corp International,-,Consumer Cyclical,Personal Services,USA,NYSE,10.66B,20.37,...,NaN,-,NaN,-,-,-,-,-,-,-
4,905,WMS,Advanced Drainage Systems Inc,-,Industrials,Building Products & Equipment,USA,NYSE,10.64B,25.49,...,NaN,-,NaN,-,-,-,-,-,-,-
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1555,956,PSO,Pearson plc ADR,-,Communication Services,Publishing,United Kingdom,NYSE,9.84B,24.83,...,NaN,-,NaN,-,-,-,-,-,-,-
1556,957,SQM,Sociedad Quimica Y Minera de Chile SA ADR,-,Basic Materials,Specialty Chemicals,Chile,NYSE,9.83B,24.20,...,NaN,-,NaN,-,-,-,-,-,-,-
1557,958,WYNN,Wynn Resorts Ltd,S&P 500,Consumer Cyclical,Resorts & Casinos,USA,NASD,9.82B,27.08,...,NaN,-,NaN,-,-,-,-,-,-,-
1558,959,VMI,Valmont Industries Inc,-,Industrials,Conglomerates,USA,NYSE,9.81B,28.05,...,NaN,-,NaN,-,-,-,-,-,-,-
